# Actividad 5.3 — Simulación Monte Carlo de inventarios EOQ

## Parte 1: análisis estadístico de la demanda

**Materia:** Modelación y simulación  
**Fuente:** archivo `5.3.- demanda_simulada_tres_ubicaciones_mexico.xlsx`, en este mismo repositorio.

### ¿Qué vamos a hacer ahora?
1. Abrir el Excel y revisar hojas, columnas y primeras observaciones.
2. Preparar las ventas/demandas diarias de las **tres ciudades de México**.
3. Calcular media, desviación estándar, mínimo, máximo, total anual y días con demanda cero.
4. Construir una **tabla de frecuencias absolutas, relativas y acumuladas por ciudad**.
5. Visualizar la distribución empírica de la demanda.

Este notebook cubre el **Paso 1.1 de la actividad**, antes de calcular EOQ, ROP, SS o simular nuevos días. **No genera números pseudoaleatorios todavía:** primero debemos estudiar los datos proporcionados.

> El archivo del repositorio se denomina «demanda_simulada...». Trabajaremos con esos registros como los datos de entrada suministrados; no los presentaremos como observaciones de ventas reales sin confirmarlo.

**Dependencias:** `pandas`, `openpyxl`, `numpy` y `matplotlib`. Si no están instaladas, utiliza `pip install pandas openpyxl numpy matplotlib` en la terminal de VS Code.

## Paso 1. Cargar las bibliotecas

Usamos `pandas` para tablas, `numpy` para operaciones numéricas, `matplotlib` para visualizar los resultados y `display` para mostrar las tablas en Jupyter.

`read_excel()` abrirá el libro que está guardado en la **misma carpeta** que este notebook.

In [ ]:
from pathlib import Path
import unicodedata

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

## Paso 2. Abrir el Excel y reconocer su estructura

Antes de hacer cálculos, es importante identificar las hojas y los nombres reales de las columnas. Las ventas podrían estar:
- En una sola hoja con columnas de **ciudad** y **demanda**.
- En una sola hoja con **una columna de demanda por ciudad**.
- En **una hoja por ciudad**.

El código de esta sección muestra la estructura sin modificar los datos.

In [ ]:
ARCHIVO = Path("5.3.- demanda_simulada_tres_ubicaciones_mexico.xlsx")

if not ARCHIVO.is_file():
    raise FileNotFoundError(
        f"No encuentro {ARCHIVO.name} en {Path.cwd()}. "
        "Abre el notebook en la carpeta raíz del repositorio y ejecuta git pull."
    )

hojas = pd.read_excel(ARCHIVO, sheet_name=None)

print(f"Archivo: {ARCHIVO.name}")
print(f"Hojas detectadas: {list(hojas)}")

for nombre_hoja, df in hojas.items():
    print(f"\nHOJA: {nombre_hoja}")
    print(f"Filas: {len(df)} | Columnas: {list(df.columns)}")
    display(df.head(5))

## Paso 3. Preparar las columnas de ciudad y demanda

La meta es crear una tabla común con **dos columnas**: `Ciudad` y `Demanda`.

La función de abajo intenta reconocer formatos frecuentes:
- **Formato largo:** las columnas identifican la ciudad y las unidades vendidas.
- **Formato ancho:** cada ciudad aparece en una columna.
- **Hojas separadas:** cada hoja contiene una ciudad y su demanda.

Si los encabezados del Excel son distintos, el programa dará una explicación del problema y podremos adaptarlo **sin inventar información**.

In [ ]:
def simplificar(nombre):
    """Normaliza los encabezados (elimina tildes, espacios y guiones)."""
    texto = unicodedata.normalize("NFKD", str(nombre))
    texto = "".join(c for c in texto if not unicodedata.combining(c))
    return "".join(c for c in texto.lower() if c.isalnum())


NOMBRES_CIUDAD = {"ciudad", "ubicacion", "sucursal", "localidad", "plaza"}
NOMBRES_DEMANDA = {
    "demanda", "demandadiaria", "ventas", "ventasdiarias",
    "unidades", "unidadesvendidas", "cantidad", "cantidadvendida",
}
NO_DEMANDA = {
    "dia", "fecha", "mes", "ano", "anio", "semana",
    "id", "numero", "indice", "precio", "costo", "periodo",
}


def preparar_registros(hojas_excel):
    """Convierte hojas de formato largo, ancho o por ciudad a Ciudad/Demanda."""
    partes = []
    hojas_omitidas = []

    for nombre_hoja, original in hojas_excel.items():
        df = original.dropna(how="all").copy()
        if df.empty:
            hojas_omitidas.append(nombre_hoja)
            continue

        etiquetas = {col: simplificar(col) for col in df.columns}
        col_ciudad = next((c for c, e in etiquetas.items() if e in NOMBRES_CIUDAD), None)
        posibles_demanda = [
            c for c, e in etiquetas.items()
            if e in NOMBRES_DEMANDA
            or e.startswith("demanda")
            or e.startswith("ventas")
            or e.startswith("unidadesvendidas")
        ]

        # Caso A: ciudad y demanda en columnas, una ciudad por fila.
        if col_ciudad is not None and posibles_demanda:
            col_demanda = posibles_demanda[0]
            parte = df[[col_ciudad, col_demanda]].copy()
            parte.columns = ["Ciudad", "Demanda"]
            partes.append(parte)
            print(f"{nombre_hoja}: formato largo (Ciudad={col_ciudad}, Demanda={col_demanda}).")
            continue

        # Caso B: cada ciudad es una columna numérica.
        # Si una hoja contiene dos o más columnas de demanda y ninguna
        # columna explícita de ciudad, se tratan como ciudades distintas.
        candidatos = []
        for col in df.columns:
            etiqueta = etiquetas[col]
            if etiqueta in NO_DEMANDA or any(
                palabra in etiqueta for palabra in ("fecha", "producto", "precio", "costo")
            ):
                continue
            valores = pd.to_numeric(df[col], errors="coerce")
            if valores.notna().sum() > 0 and valores.notna().sum() >= 0.8 * df[col].notna().sum():
                candidatos.append(col)

        if col_ciudad is None and len(candidatos) >= 2:
            columnas = posibles_demanda if len(posibles_demanda) >= 2 else candidatos
            parte = df[columnas].melt(var_name="Ciudad", value_name="Demanda")
            partes.append(parte)
            print(f"{nombre_hoja}: formato ancho; columnas {columnas}.")
            continue

        # Caso C: hoja individual con una sola columna de demanda.
        if col_ciudad is None and posibles_demanda:
            col = posibles_demanda[0]
            parte = pd.DataFrame({"Ciudad": nombre_hoja, "Demanda": df[col]})
            partes.append(parte)
            print(f"{nombre_hoja}: hoja individual, demanda en {col}.")
            continue
        if col_ciudad is None and len(candidatos) == 1:
            col = candidatos[0]
            parte = pd.DataFrame({"Ciudad": nombre_hoja, "Demanda": df[col]})
            partes.append(parte)
            print(f"{nombre_hoja}: hoja individual, demanda en {col}.")
            continue

        hojas_omitidas.append(nombre_hoja)

    if not partes:
        raise ValueError(
            "No se reconocieron columnas de ciudad y demanda. "
            "Revisa los encabezados impresos en el Paso 2 para adaptar la función."
        )

    registros = pd.concat(partes, ignore_index=True)
    registros = registros.dropna(subset=["Ciudad", "Demanda"]).copy()
    registros["Ciudad"] = registros["Ciudad"].astype(str).str.strip()
    registros["Demanda"] = pd.to_numeric(registros["Demanda"], errors="coerce")

    if registros["Demanda"].isna().any():
        raise ValueError("Hay demandas no numéricas. Revisa los datos originales.")
    if (registros["Demanda"] < 0).any():
        raise ValueError("Hay demandas negativas: revisa su significado antes de continuar.")
    if not np.allclose(registros["Demanda"], registros["Demanda"].round()):
        raise ValueError("Se detectaron demandas fraccionarias: revisar unidades.")
    registros["Demanda"] = registros["Demanda"].astype(int)

    if hojas_omitidas:
        print("Hojas omitidas por no reconocer datos de demanda:", hojas_omitidas)

    return registros


registros = preparar_registros(hojas)
print("\nVista de los datos estandarizados:")
display(registros.head(12))
print("\nObservaciones por ciudad:")
display(registros.groupby("Ciudad").size().rename("Días registrados").to_frame())

if registros["Ciudad"].nunique() != 3:
    print("AVISO: la actividad requiere tres ciudades; revisa la detección antes de continuar.")

## Paso 4. Estadísticas descriptivas de la demanda

De acuerdo con la actividad, para cada ciudad necesitamos:

- **Media diaria:** promedio de la demanda observada.
- **Desviación estándar:** cuánto varía respecto al promedio.
- **Mínima y máxima:** extremos de demanda registrados.
- **Demanda total del periodo:** suma de todas las demandas diarias.
- **Días con demanda cero:** ocasiones en que no hubo demanda.

La actividad pide un año de datos. Por ello, también mostramos **cuántos días hay por ciudad** para verificar la cobertura del archivo. No suponemos que hay exactamente 365 días sin comprobarlo.

In [ ]:
resumen = (
    registros.groupby("Ciudad")["Demanda"]
    .agg(
        Dias_registrados="count",
        Demanda_media="mean",
        Desviacion_estandar="std",
        Demanda_minima="min",
        Demanda_maxima="max",
        Demanda_total="sum",
        Dias_demanda_cero=lambda s: (s == 0).sum(),
    )
    .reset_index()
)

display(resumen.round(2))
print("Recuerda: para interpretar 'Demanda_total' como anual, confirma que el Excel")
print("contenga el año completo en cada ciudad.")

## Paso 5. Tablas de frecuencias empíricas (una por ciudad)

Construimos las columnas solicitadas en el PDF:

- **Demanda:** número de unidades demandadas en el día.
- **Frecuencia absoluta:** número de días con esa demanda.
- **Frecuencia relativa:** frecuencia absoluta dividida entre los días registrados.
- **Frecuencia acumulada:** suma progresiva de las frecuencias absolutas.
- **Frecuencia relativa acumulada:** suma progresiva de las probabilidades.

La última columna será la clave para **transformar números pseudoaleatorios U(0,1) en demandas simuladas**, en la siguiente parte del proyecto.

In [ ]:
tablas_frecuencia = {}

for ciudad, grupo in registros.groupby("Ciudad"):
    tabla = (
        grupo["Demanda"]
        .value_counts()
        .sort_index()
        .rename_axis("Demanda")
        .reset_index(name="Frecuencia absoluta")
    )
    total = tabla["Frecuencia absoluta"].sum()
    tabla["Frecuencia relativa"] = tabla["Frecuencia absoluta"] / total
    tabla["Frecuencia acumulada"] = tabla["Frecuencia absoluta"].cumsum()
    tabla["Frecuencia relativa acumulada"] = tabla["Frecuencia relativa"].cumsum()

    tablas_frecuencia[ciudad] = tabla

    print(f"\nTABLA DE FRECUENCIAS: {ciudad} ({total} días)")
    display(tabla.round(5))
    assert tabla["Frecuencia absoluta"].sum() == total
    assert np.isclose(tabla["Frecuencia relativa acumulada"].iloc[-1], 1.0)

print("\nComprobación: todas las frecuencias relativas acumuladas terminan en 1.")

## Paso 6. Visualizar la distribución de demanda de cada ciudad

Un histograma nos permite observar qué valores de demanda son más frecuentes y comparar visualmente las ciudades.

**Ojo:** aquí visualizamos la *demanda registrada en el Excel*, no la demanda generada por Monte Carlo. Eso llegará en el siguiente bloque del proyecto.

In [ ]:
for ciudad, grupo in registros.groupby("Ciudad"):
    fig, ax = plt.subplots(figsize=(9, 3.5))
    ax.hist(grupo["Demanda"], bins="auto", edgecolor="black")
    ax.set_title(f"Demanda diaria — {ciudad}")
    ax.set_xlabel("Unidades demandadas por día")
    ax.set_ylabel("Número de días")
    ax.grid(axis="y", alpha=0.2)
    plt.tight_layout()
    plt.show()

## ¿Qué tenemos listo y qué falta?

**Parte completada en este notebook:**
- Carga y revisión de los datos del Excel.
- Identificación de ciudades y demanda diaria.
- Estadística descriptiva por ciudad.
- Frecuencias absolutas, relativas y acumuladas.
- Histogramas para entender la demanda.

**Siguiente etapa (todavía no implementada):**
1. Solicitar por ciudad los parámetros del modelo EOQ: costo de ordenar (S), costo de mantener (H), plazo de entrega (L), inventario inicial y nivel de servicio.
2. Calcular **Q\***, **ROP** y **SS**.
3. Generar números uniformes U y transformarlos en demandas diarias con las tablas acumuladas.
4. Simular n días, registrar los pedidos y elaborar indicadores y gráficas.

> Los costos y demás parámetros deben introducirse a partir de las condiciones del problema; **no se inventan** en esta primera etapa.

### Relación con el método de Monte Carlo

Si la frecuencia relativa acumulada de cierta ciudad es 0.30 para demanda 10, y 0.70 para demanda 20, un número uniforme U = 0.56 representa **20 unidades**. Así utilizaremos las distribuciones empíricas que acabamos de construir.